# Voicers TTS vs Whisper STT Comparison

Generate audio with voicers, transcribe with Whisper, compare what was said vs what was heard.

In [1]:
import subprocess, json, os

test_phrases = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog",
    "How are you doing today?",
    "Welcome to the voice synthesis engine",
    "She sells seashells by the seashore",
]

output_dir = "/tmp/voicers_stt_test"
os.makedirs(output_dir, exist_ok=True)

# Generate audio for each phrase
for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    result = subprocess.run(
        [
            "cargo",
            "run",
            "--release",
            "-p",
            "voicers-cli",
            "--",
            "--text",
            phrase,
            "--output",
            wav_path,
        ],
        capture_output=True,
        text=True,
        cwd="/Users/kylekelley/conductor/workspaces/voicers/algiers",
    )
    print(f"[{i}] '{phrase}' -> {wav_path}")
    if result.stderr:
        # Show the phoneme chunks from G2P
        for line in result.stderr.strip().split("\n"):
            if "chunk" in line.lower():
                print(f"    {line.strip()}")
    if result.returncode != 0:
        print(f"    ERROR: {result.stderr[-200:]}")

print(f"\nGenerated {len(test_phrases)} audio files in {output_dir}")

FileNotFoundError: [Errno 2] No such file or directory: 'cargo'

In [1]:
import subprocess, os

VOICERS = (
    "/Users/kylekelley/conductor/workspaces/voicers/algiers/target/release/voicers-cli"
)

test_phrases = [
    "Hello world",
    "The quick brown fox jumps over the lazy dog",
    "How are you doing today?",
    "Welcome to the voice synthesis engine",
    "She sells seashells by the seashore",
]

output_dir = "/tmp/voicers_stt_test"
os.makedirs(output_dir, exist_ok=True)

env = os.environ.copy()
env["PATH"] = "/opt/homebrew/bin:" + env.get("PATH", "")

for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    result = subprocess.run(
        [VOICERS, "--text", phrase, "--output", wav_path],
        capture_output=True,
        text=True,
        env=env,
    )
    status = "ok" if result.returncode == 0 else "FAIL"
    print(f"[{i}] {status} | '{phrase}' -> {wav_path}")
    for line in result.stderr.strip().split("\n"):
        if "chunk" in line.lower():
            print(f"    {line.strip()}")
    if result.returncode != 0:
        print(f"    {result.stderr[-300:]}")

print(f"\nDone: {len(test_phrases)} files in {output_dir}")

[0] ok | 'Hello world' -> /tmp/voicers_stt_test/phrase_0.wav
    chunk 1: həlˈO wˈɜɹld
[1] ok | 'The quick brown fox jumps over the lazy dog' -> /tmp/voicers_stt_test/
phrase_1.wav
    chunk 1: ðə kwˈɪk bɹˈWn fˈɑks ʤˈʌmps ˌOvɚ ðə lˈAzi dˈɑɡ
[2] ok | 'How are you doing today?' -> /tmp/voicers_stt_test/phrase_2.wav
    chunk 1: hˌW ɑɹ ju dˌuɪŋ tədˈA
[3] ok | 'Welcome to the voice synthesis engine' -> /tmp/voicers_stt_test/phrase
_3.wav
    chunk 1: wˈɛlkʌm tə ðə vˈYs sˈɪnθəsˌɪs ˈɛnʤɪn
[4] ok | 'She sells seashells by the seashore' -> /tmp/voicers_stt_test/phrase_4
.wav
    chunk 1: ʃi sˈɛlz sˈiʃɛlz bI ðə sˈiʃɔɹ

Done: 5 files in /tmp/voicers_stt_test


In [4]:
import mlx_whisper

# Transcribe each generated file
results = []
for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    transcription = mlx_whisper.transcribe(
        wav_path, path_or_hf_repo="mlx-community/whisper-small"
    )
    heard = transcription["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?,") == phrase.lower().rstrip(".!?,")
        else "DIFF"
    )
    results.append((phrase, heard, match))
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()

RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69b9f19b-7649795313cee07e54fcd413;21da5744-cf23-45cc-8eb0-fe2d4703437c)

Repository Not Found for url: https://huggingface.co/api/models/mlx-community/whisper-small/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication

In [5]:
import mlx_whisper

WHISPER_MODEL = "mlx-community/whisper-large-v3-turbo-asr-fp16"

results = []
for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    transcription = mlx_whisper.transcribe(wav_path, path_or_hf_repo=WHISPER_MODEL)
    heard = transcription["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?, ") == phrase.lower().rstrip(".!?, ")
        else "DIFF"
    )
    results.append((phrase, heard, match))
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

TypeError: ModelDimensions.__init__() got an unexpected keyword argument 'activation_dropout'

In [3]:
import mlx_whisper

WHISPER_MODEL = "mlx-community/whisper-large-v3-turbo"

results = []
for i, phrase in enumerate(test_phrases):
    wav_path = f"{output_dir}/phrase_{i}.wav"
    transcription = mlx_whisper.transcribe(wav_path, path_or_hf_repo=WHISPER_MODEL)
    heard = transcription["text"].strip()
    match = (
        "MATCH"
        if heard.lower().rstrip(".!?, ") == phrase.lower().rstrip(".!?, ")
        else "DIFF"
    )
    results.append((phrase, heard, match))
    print(f"[{i}] {match}")
    print(f"    Sent:  {phrase}")
    print(f"    Heard: {heard}")
    print()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[0] DIFF
    Sent:  Hello world
    Heard: Hello, world.

[1] DIFF
    Sent:  The quick brown fox jumps over the lazy dog
    Heard: the quick brown foxes over the lazy dog again.

[2] MATCH
    Sent:  How are you doing today?
    Heard: How are you doing today?

[3] DIFF
    Sent:  Welcome to the voice synthesis engine
    Heard: Well, come to the voice synthesis engine

[4] DIFF
    Sent:  She sells seashells by the seashore
    Heard: J-E-Selsa-Z-S-S-S-Elza-Bai-I-The-Z-S-S-S-Or-
